# ATAC-seq Peak CNN

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from pyfaidx import Fasta
from tqdm.notebook import tqdm
import random

In [ ]:
bed = pd.read_csv("ENCFF748UZH.bed", sep="\t", header=None)
bed.columns = [
    "chrom",
    "start",
    "end",
    "name",
    "score",
    "strand",
    "signalValue",
    "pValue",
    "qValue",
    "peak"
]
bed["length"] = bed["end"] - bed["start"]
bed.head()

In [ ]:
BED_FILE = "bed"
FASTA_FILE = "hg38.fa"
WINDOW_SIZE = 200
BATCH_SIZE = 256
EPOCHS = 20
LR = 1e-3
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

VALID_CHROMS = {f"chr{i}" for i in list(range(1, 23)) + ["X", "Y"]}

HG38_CHROM_SIZES = {
    "chr1": 248956422, "chr2": 242193529, "chr3": 198295559,
    "chr4": 190214555, "chr5": 181538259, "chr6": 170805979,
    "chr7": 159345973, "chr8": 145138636, "chr9": 138394717,
    "chr10": 133797422, "chr11": 135086622, "chr12": 133275309,
    "chr13": 114364328, "chr14": 107043718, "chr15": 101991189,
    "chr16": 90338345,  "chr17": 83257441,  "chr18": 80373285,
    "chr19": 58617616,  "chr20": 64444167,  "chr21": 46709983,
    "chr22": 50818468,  "chrX": 156040895,  "chrY": 57227415,
}

Loading Peaks

In [ ]:
from pyfaidx import Fasta
fasta = Fasta("hg38.fa")
print(list(fasta.keys())[:5])

In [ ]:
def load_peaks(df):
    df = df.copy()
    df = df[df["chrom"].isin(VALID_CHROMS)]
    df["summit"] = df["start"] + df["peak"]
    half = WINDOW_SIZE // 2
    df["win_start"] = df["summit"] - half
    df["win_end"]   = df["summit"] + half
    df = df[df["win_start"] >= 0]
    valid_rows = []
    for _, row in df.iterrows():
        chrom_size = HG38_CHROM_SIZES.get(row["chrom"], 0)
        if row["win_end"] <= chrom_size:
            valid_rows.append(row)
    df = pd.DataFrame(valid_rows)
    return df[["chrom", "win_start", "win_end"]].reset_index(drop=True)

peaks = load_peaks(bed)
peaks.head()

Generating non-peaks

In [ ]:
def generate_negatives(peaks_df, n_negatives=None):
    if n_negatives is None:
        n_negatives = len(peaks_df)
    peak_set = {}
    for _, row in peaks_df.iterrows():
        peak_set.setdefault(row["chrom"], []).append((row["win_start"], row["win_end"]))

    def overlaps(chrom, start, end):
        for ps, pe in peak_set.get(chrom, []):
            if start < pe and end > ps:
                return True
        return False

    chroms = list(VALID_CHROMS & set(HG38_CHROM_SIZES))
    weights = np.array([HG38_CHROM_SIZES[c] for c in chroms], dtype=float)
    weights /= weights.sum()

    half = WINDOW_SIZE // 2
    negatives, attempts = [], 0

    with tqdm(total=n_negatives, desc="Sampling negatives") as pbar:
        while len(negatives) < n_negatives and attempts < n_negatives * 20:
            chrom = np.random.choice(chroms, p=weights)
            summit = random.randint(half, HG38_CHROM_SIZES[chrom] - half)
            s, e = summit - half, summit + half
            if not overlaps(chrom, s, e):
                negatives.append({"chrom": chrom, "win_start": s, "win_end": e})
                pbar.update(1)
            attempts += 1
    return pd.DataFrame(negatives)

negatives = generate_negatives(peaks)
negatives.head()

One-hot Encoding

In [ ]:
NUCLEOTIDE_MAP = {"A": 0, "C": 1, "G": 2, "T": 3}

def one_hot_encode(sequence):
    seq = sequence.upper()
    encoding = np.zeros((len(seq), 4), dtype=np.float32)
    for i, base in enumerate(seq):
        if base in NUCLEOTIDE_MAP:
            encoding[i, NUCLEOTIDE_MAP[base]] = 1.0
    return encoding


def fetch_and_encode(regions_df, fasta, desc="Fetching"):
    sequences, failed = [], 0
    for _, row in tqdm(regions_df.iterrows(), total=len(regions_df), desc=desc):
        try:
            seq = fasta[row["chrom"]][row["win_start"]:row["win_end"]].seq
            if len(seq) == WINDOW_SIZE:
                sequences.append(one_hot_encode(seq))
            else:
                sequences.append(None); failed += 1
        except Exception:
            sequences.append(None); failed += 1
    if failed:
        print(f"  Warning: {failed} sequences failed (will be dropped)")
    return sequences


fasta = Fasta(FASTA_FILE)

pos_seqs = fetch_and_encode(peaks, fasta, desc="Positive seqs")
neg_seqs = fetch_and_encode(negatives, fasta, desc="Negative seqs")

pos_seqs = [s for s in pos_seqs if s is not None]
neg_seqs = [s for s in neg_seqs if s is not None]

Train/test Split

In [ ]:
X = np.array(pos_seqs + neg_seqs, dtype=np.float32)   # (N, 200, 4)
y = np.array([1] * len(pos_seqs) + [0] * len(neg_seqs), dtype=np.float32)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp)

Building CNN

In [ ]:
def build_model(seq_len=WINDOW_SIZE, learning_rate=LR)
    inputs = keras.Input(shape=(seq_len, 4), name="sequence")
    x = layers.Conv1D(64,  kernel_size=8, padding="same", activation="relu", name="conv1")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv1D(128, kernel_size=8, padding="same", activation="relu", name="conv2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv1D(256, kernel_size=8, padding="same", activation="relu", name="conv3")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64,  activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = keras.Model(inputs, outputs, name="ATACNet")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auroc"),
                              keras.metrics.AUC(curve="PR", name="auprc")]
    )
    return model


model = build_model()
model.summary()

Train Model

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        "best_atac_cnn.keras",
        monitor="val_auroc",
        mode="max",
        save_best_only=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auroc",
        mode="max",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

Plotting

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric, title in zip(
    axes,
    ["loss", "auroc", "auprc"],
    ["Loss", "AUROC", "AUPRC"]
):
    ax.plot(history.history[metric],     label="Train")
    ax.plot(history.history[f"val_{metric}"], label="Val")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Evaluation

In [ ]:
best_model = keras.models.load_model("best_atac_cnn.keras")

test_preds = best_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).flatten()

test_auroc = roc_auc_score(y_test, test_preds)
test_auprc = average_precision_score(y_test, test_preds)
test_acc   = ((test_preds > 0.5) == y_test).mean()

print("══════════════════════════════")
print("     Test Set Results         ")
print("══════════════════════════════")
print(f"  Accuracy : {test_acc:.4f}")
print(f"  AUROC    : {test_auroc:.4f}   (1.0 = perfect, 0.5 = random)")
print(f"  AUPRC    : {test_auprc:.4f}   (higher = better)")
print("══════════════════════════════")